<a href="https://colab.research.google.com/github/pdjota/arty/blob/main/Arty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Claude conversation
https://claude.ai/chat/ab7a9ed3-f964-40d4-a1b8-addca890d6bb

In [1]:
!pip install git+https://github.com/openai/CLIP.git -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00


In [2]:
import os
import torch
import clip
import numpy as np
from datasets import load_dataset, Dataset, load_from_disk
from google.colab import drive
from PIL import Image

drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/wikiart_1k_features"
LOCAL_PATH = "/content/wikiart_1k_features"

# Load CLIP
device = ""
if torch.cuda.is_available():
  print("cuda device available")
  device = "cuda"
else:
  device = "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

if os.path.exists(LOCAL_PATH):
    data = torch.load(f"{LOCAL_PATH}/data.pt")
    print(f"Loaded from local cache ✅ ({len(data['features'])} samples)")

elif os.path.exists(DRIVE_PATH):
    print("Loading from Drive...")
    import shutil
    shutil.copytree(DRIVE_PATH, LOCAL_PATH)
    data = torch.load(f"{LOCAL_PATH}/data.pt")
    print(f"Loaded from Drive ✅ ({len(data['features'])} samples)")

else:
    print("First time! Downloading + extracting features...")
    streamed = load_dataset("huggan/wikiart", streaming=True, split="train")
    streamed = streamed.take(1000)

    features, artists, genres, styles = [], [], [], []

    with torch.no_grad():
        for i, sample in enumerate(streamed):
            # Full size image → CLIP preprocess (224x224 internally)
            img = preprocess(sample['image']).unsqueeze(0).to(device)
            feat = model.encode_image(img).cpu().squeeze()
            features.append(feat)
            artists.append(sample['artist'])
            genres.append(sample['genre'])
            styles.append(sample['style'])

            if i % 100 == 0:
                print(f"  {i}/1000 processed...")

    data = {
        'features': torch.stack(features),  # (1000, 512)
        'artist': artists,
        'genre': genres,
        'style': styles
    }

    os.makedirs(LOCAL_PATH, exist_ok=True)
    os.makedirs(DRIVE_PATH, exist_ok=True)
    torch.save(data, f"{LOCAL_PATH}/data.pt")
    torch.save(data, f"{DRIVE_PATH}/data.pt")
    print(f"Features extracted + saved ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cuda device available


100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 213MiB/s]


First time! Downloading + extracting features...


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

  0/1000 processed...
  100/1000 processed...
  200/1000 processed...
  300/1000 processed...
  400/1000 processed...
  500/1000 processed...
  600/1000 processed...
  700/1000 processed...
  800/1000 processed...
  900/1000 processed...
Features extracted + saved ✅
